# Notebook 8: Final Evaluation

The earlier notebooks each measured one change. This one measures the whole system on 30 new questions, written after this code was committed.

Three things are measured separately, because a bad final answer doesn't say where it went wrong:

1. **Retrieval:** is the sentence holding the answer in the top 5?
2. **Generation:** given exactly the right sentence, does the model answer correctly?
3. **End to end:** SQL-only, vector-only and hybrid, each answer scored right, wrong or refused.

Each question runs once at `temperature=0`. The conclusions cover how repeatable that turned out to be.

In [ ]:
import json
import re
import sys
from collections import Counter
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.retriever import bm25_search, dense_search, fused_search, sentences
from src.router import answer_from_context, answer_vector, ask, route, usable
from src.text_to_sql import NO_SQL, con, numbers_in, run_sql, sql_answer

with open(PROJECT_ROOT / "eval" / "questions.json", encoding="utf-8") as f:
    FINAL = [q for q in json.load(f) if q["split"] == "final"]

print(f"Final questions: {len(FINAL)}", Counter(q["kind"] for q in FINAL))

## Scoring

This fixes two faults found in Notebook 6b. Number words now count, so "Seven races" matches 7. When Python falls back to raw rows, only the top row counts as the answer, so a list can't score by luck.

In [ ]:
REFUSAL = re.compile(r"(not|isn't) in the (context|database|data)|no usable result|no information|no data|"
                     r"cannot (provide|answer|determine)|can't answer|not mentioned|no .{0,30} mentioned|"
                     r"returned no rows|not provided|does not (state|contain|mention)", re.IGNORECASE)
NUMBER_WORDS = {"zero": 0, "one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "seven": 7, "eight": 8,
                "nine": 9, "ten": 10, "eleven": 11, "twelve": 12}


def expected(q: dict) -> list:
    values = list(q.get("accept", []))
    if q["truth_sql"]:
        values.insert(0, con.execute(q["truth_sql"]).fetchone()[0])
    return values


def text_has(answer: str, value) -> bool:
    if isinstance(value, str):
        return value.lower() in answer.lower()
    found = numbers_in(answer) | {NUMBER_WORDS[w] for w in re.findall(r"[a-z]+", answer.lower()) if w in NUMBER_WORDS}
    return any(abs(n - float(value)) <= 0.01 for n in found)


def outcome(q: dict, answer: str) -> str:
    """right, refused, or wrong. A wrong answer is the worst result, because it looks true."""
    refused = bool(REFUSAL.search(answer))
    values = expected(q)
    if not values:
        return "right" if refused else "wrong"
    if refused:
        return "refused"
    if answer.startswith("Result of the query:"):  # raw rows: only the top row is the answer
        lines = answer.splitlines()
        answer = lines[2] if len(lines) > 2 else ""
    return "right" if any(text_has(answer, v) for v in values) else "wrong"


def tally(rows: list, name: str) -> dict:
    return {k: sum(r[name] == k for r in rows) for k in ("right", "wrong", "refused")}

## Retrieval alone

Only questions answered by one sentence are scored here. Aggregates like career wins aren't in any single sentence, so retrieval can't answer them by design.

In [ ]:
RETRIEVABLE = [q for q in FINAL if q.get("evidence")]


def holds_evidence(sentence: str, q: dict) -> bool:
    return all(term in sentence for term in q["evidence"])


def gold_sentence(q: dict) -> str:
    matches = [s for s in sentences if holds_evidence(s, q)]
    assert len(matches) == 1, (q["id"], len(matches))
    return matches[0]


searches = {"dense": dense_search, "bm25": bm25_search, "fused": fused_search}
retrieval = {name: {} for name in searches}
for q in RETRIEVABLE:
    for name, search in searches.items():
        retrieval[name][q["id"]] = any(holds_evidence(sentences[i], q) for i in search(q["question"], 5))

for name, hits in retrieval.items():
    print(f"{name:<6} found the answer sentence in the top 5: {sum(hits.values())}/{len(hits)}")
print("fused misses:", [k for k, v in retrieval["fused"].items() if not v])

## Generation alone

In [ ]:
# The right sentence is handed over, so any miss here is the model's, not retrieval's.
generation = {}
for q in RETRIEVABLE:
    answer = answer_from_context(q["question"], [gold_sentence(q)])
    generation[q["id"]] = {"outcome": outcome(q, answer), "answer": answer}

print("Generation with the right sentence:", Counter(g["outcome"] for g in generation.values()))
for key, g in generation.items():
    if g["outcome"] != "right":
        print(f"  {key} [{g['outcome']}] {g['answer'][:110]!r}")

## End to end

`query_right` records whether the SQL's top row held the answer, before the model wrote a sentence about it. That separates a bad query from a bad sentence.

In [ ]:
def first_row_right(q: dict, run: dict) -> bool:
    """Was the query itself right, before the model wrote a sentence about it?"""
    if not expected(q) or run["error"] or run["df"] is None or run["df"].empty:
        return False
    row = " ".join(str(cell) for cell in run["df"].iloc[0])
    return any(text_has(row, v) for v in expected(q))


rows = []
for q in FINAL:
    run = run_sql(q["question"])
    if run["sql"] == NO_SQL:
        sql_text = "That isn't in the database."
    elif usable(run):
        sql_text = sql_answer(q["question"], run["df"])
    else:
        sql_text = "No usable result from the database."
    vector_text = answer_vector(q["question"])[0]
    hybrid = ask(q["question"])
    rows.append({"id": q["id"], "kind": q["kind"], "question": q["question"], "route": route(q["question"]),
                 "path": hybrid["path"], "sql": outcome(q, sql_text), "vector": outcome(q, vector_text),
                 "hybrid": outcome(q, hybrid["answer"]), "query_right": first_row_right(q, run), "generated_sql": run["sql"],
                 "sql_answer": sql_text, "vector_answer": vector_text, "hybrid_answer": hybrid["answer"]})

for name in ("sql", "vector", "hybrid"):
    print(f"{name:<7} {tally(rows, name)}")
print()
for kind in dict.fromkeys(r["kind"] for r in rows):
    subset = [r for r in rows if r["kind"] == kind]
    print(f"{kind:<13} of {len(subset):>2}:", {n: tally(subset, n)["right"] for n in ("sql", "vector", "hybrid")})

In [ ]:
answerable = [r for r in rows if r["kind"] != "unanswerable"]
print(f"SQL queries right on answerable questions: {sum(r['query_right'] for r in answerable)}/{len(answerable)}")
print(f"Right query, wrong sentence: {[r['id'] for r in answerable if r['query_right'] and r['sql'] != 'right']}")
print(f"Routed as expected: {sum(r['route'] == q['route'] for r, q in zip(rows, FINAL))}/{len(rows)}")
print()
for r in rows:
    print(f"{r['id']} {r['kind']:<12} {r['path']:<10} S={r['sql']:<7} V={r['vector']:<7} H={r['hybrid']:<7} {r['hybrid_answer'][:80]!r}")

out = PROJECT_ROOT / "eval" / "results_nb08.json"
out.write_text(json.dumps({"retrieval": retrieval, "generation": generation, "rows": rows}, indent=2, ensure_ascii=False), encoding="utf-8")

## What We Learned

30 new questions, written after this code was committed, with live 2026 data through round 18.

| Pipeline | Right | Wrong | Refused |
|---|---|---|---|
| SQL only | 19 | 6 | 5 |
| Vector only | 16 | 4 | 10 |
| Hybrid | 21 | 5 | 4 |

| Kind | SQL | Vector | Hybrid |
|---|---|---|---|
| single fact (10) | 8 | 4 | 7 |
| aggregate (8) | 5 | 1 | 5 |
| narrative (5) | 1 | 4 | 4 |
| unanswerable (7) | 5 | 7 | 5 |

### Findings

1. **Hybrid is the best pipeline,** at 21 of 30. It takes SQL's strength on aggregates and vector's on narratives.
2. **Retrieval, not the model, limits the vector path.** Given the right sentence, the model answered 14 of 15. Fused retrieval found that sentence for 9 of 15 (dense 7, BM25 8).
3. **The vocabulary mismatch from Notebook 4 survives.** Four of the six retrieval misses are "who won" or "on pole" questions, and the corpus says "1st (winner)" and "starting from position 1". Vague race questions also fail, because every driver's sentence looks equally relevant.
4. **Wrong answers are the real cost.** "Fernando Alonso won 17 distinct races" (truth 32) ran cleanly and used only the question's own words, so no check caught it.
5. **Hybrid still loses on fact-shaped unanswerable questions.** Vector refused all 7; hybrid refused 5. "Verstappen led 7 distinct grid positions" is the router sending a question to a path that can't refuse.

### Known problems

- `SAFE_NUMBERS` allows `1`, so `round = 1` passes the value guard. That sent the 2003 Canadian Grand Prix to an empty result.
- Some generated SQL came back with no `FROM`, costing a retry. It didn't reproduce, so the cause is unconfirmed. `clean_sql` splitting on a blank line is the likeliest suspect.
- `final_07` was right through SQL-only and fell back in hybrid, despite identical code. See the note below.

### Limits of this evaluation

- 30 questions, each run once, so a gap of two isn't conclusive.
- **Runs are not fully repeatable.** `temperature=0` gave identical output within a short test, but the same question changed between runs and sessions. The cause is unconfirmed; earlier prompts affecting model state is a guess. Scores carry some noise.
- final_25 ("not specified in the provided data") was a valid refusal the scorer missed, so hybrid is arguably 22 right and 4 wrong. The recorded score stays at 21.
- This set has now been seen, so any further change needs new questions.
